# 06 · Live Demo  🌿
**Run under pressure:** (1) run the two preamble cells, (2) load the predictor, (3) set `DEMO_IMAGES` (a folder or list of paths) and run. Loads only saved checkpoints — no training state.

> Needs trained checkpoints on this runtime (same session as training, or a Drive-mounted `checkpoint_dir`).

In [ ]:
# === Preamble 1/2: environment & GPU report ===
# This is a REMOTE Colab kernel — it cannot see your local files.
import sys
print('Python:', sys.version.split()[0])
try:
    import torch
    print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('bfloat16 supported:', torch.cuda.is_bf16_supported())
    else:
        print('No GPU — Runtime > Change runtime type > A100 (or L4).')
except ImportError:
    print('torch installs in the next cell.')

In [ ]:
# === Preamble 2/2: clone-or-pull + install (+ optional autoreload) ===
import os, subprocess, sys

REPO_URL = "https://github.com/Kidhurshan/plant-leaf-classifier.git"  # <-- EDIT to your repo
REPO_DIR = "/content/plant-leaf-classifier"
# Private repo? use https://<TOKEN>@github.com/Kidhurshan/plant-leaf-classifier.git

if not os.path.isdir(REPO_DIR):
    print('Cloning', REPO_URL)
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                'requirements.txt'], check=True)

# Hot-reload src/ after a `git pull` (optional convenience; never fatal).
try:
    from IPython import get_ipython
    _ip = get_ipython()
    _ip.run_line_magic('load_ext', 'autoreload')
    _ip.run_line_magic('autoreload', '2')
    print('autoreload enabled.')
except Exception as _e:
    print('autoreload not enabled (non-fatal):', repr(_e))

from src.utils import sync_repo, gpu_report
sync_repo()   # git pull + print the commit hash these results are traceable to
gpu_report()

## (Optional) mount Drive if checkpoints live there

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
CKPT_DIR = None  # e.g. '/content/drive/MyDrive/task4_ckpts', else default

## Load the predictor (best single model or the ensemble)

In [ ]:
from src.config import load_config
from src.inference import build_predictor
import torch

cfg = load_config('configs/default.yaml')
if CKPT_DIR: cfg.paths.checkpoint_dir = CKPT_DIR
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
WHICH = 'best'   # 'best' | 'ensemble' | a model key
try:
    predictor = build_predictor(cfg, device, which=WHICH)
    print('Loaded:', list(predictor.models), '| classes:', predictor.class_names)
except FileNotFoundError as e:
    print('No checkpoints found:\n', e)

## Predict on your images
Set `DEMO_IMAGES` to a folder or a list of image paths.

In [ ]:
import matplotlib.pyplot as plt
DEMO_IMAGES = ['/content/plant-leaf-classifier/data']  # a folder or a list of image paths

def show(results):
    for r in results:
        cols = 2 if 'gradcam' in r else 1
        fig, ax = plt.subplots(1, cols, figsize=(3.2*cols, 3.2))
        ax = [ax] if cols == 1 else ax
        ax[0].imshow(r['image']); ax[0].axis('off')
        ax[0].set_title(f"{r['pred']}  {r['confidence']*100:.1f}%")
        if 'gradcam' in r:
            ax[1].imshow(r['gradcam']); ax[1].axis('off'); ax[1].set_title('Grad-CAM')
        plt.show()
        print('  top-3:', ', '.join(f'{n} {p*100:.1f}%' for n, p in r['topk']))

results = predictor.predict(DEMO_IMAGES, top_k=3, with_gradcam=True)
show(results[:5])

## Evaluator mode — 5 random images

In [ ]:
import random
from src.inference import gather_image_paths
pool = gather_image_paths([cfg.paths.data_dir])
if pool:
    show(predictor.predict(random.sample(pool, min(5, len(pool))),
                           top_k=3, with_gradcam=True))
else:
    print('No images under', cfg.paths.data_dir, '— point DEMO_IMAGES at real files.')

---
### ⚠️ When finished: disconnect and DELETE the runtime
`Runtime > Disconnect and delete runtime`. Colab compute units are consumed the whole time a runtime is connected.